# Red Neuronal con PyTorch — Reconocimiento de Dígitos Kannada MNIST

**Dataset:** Kannada MNIST  
**Fuente:** https://www.kaggle.com/c/Kannada-MNIST/data

## Variable objetivo (Y) — 10 clases
Dígitos del **0 al 9** del sistema numérico Kannada.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from matplotlib import pyplot
from sklearn.metrics import accuracy_score

%matplotlib inline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Carga y Preprocesamiento con Pandas

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Intelegencia Artificial/Dataset/mnist_train.csv', sep=';')

print(f'Dimensiones: {data.shape}')
print(data['label'].value_counts().sort_index())

In [ ]:
fig, axes = pyplot.subplots(2, 5, figsize=(14, 6))
for digito in range(10):
    ax      = axes[digito // 5][digito % 5]
    muestra = data[data['label'] == digito].iloc[0, 1:].values
    ax.imshow(muestra.reshape(28, 28), cmap='gray')
    ax.set_title(f'Dígito: {digito}')
    ax.axis('off')
pyplot.tight_layout()
pyplot.show()

In [ ]:
N_POR_CLASE = 5000

data_bal = pd.concat([
    data[data['label'] == d].sample(n=N_POR_CLASE, random_state=42)
    for d in range(10)
]).sample(frac=1, random_state=42).reset_index(drop=True)

pixel_cols = [c for c in data_bal.columns if c != 'label']

X = data_bal[pixel_cols].values.astype(np.float32)
y = data_bal['label'].values.astype(np.int64)
m = len(y)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

## 2. Normalización y Split 80/20

In [ ]:
# se convierten los arrays de numpy a tensores de pytorch
X_train = X[:int(0.8*m)] / 255.
X_test  = X[int(0.8*m):] / 255.
y_train = y[:int(0.8*m)]
y_test  = y[int(0.8*m):]

print(f'Entrenamiento: {len(y_train):,}')
print(f'Validación   : {len(y_test):,}')

In [ ]:
# se convierten los arrays de numpy a tensores de pytorch
X_t = torch.from_numpy(X_train).float()
Y_t = torch.from_numpy(y_train).long()

print(f'X_t shape: {X_t.shape}')
print(f'Y_t shape: {Y_t.shape}')

## 3. Definir el Modelo

Se crea una clase que hereda de `torch.nn.Module`.
En `__init__` se definen las capas y en `forward` la lógica de cálculo.

In [ ]:
# creamos una clase que hereda de torch.nn.Module
class ModeloPersonalizado(nn.Module):

    # constructor
    def __init__(self, D_in, H, D_out):

        # llamamos al constructor de la clase madre
        super(ModeloPersonalizado, self).__init__()

        # definimos nuestras capas
        self.fc1  = nn.Linear(D_in, H)
        self.relu = nn.ReLU()
        self.fc2  = nn.Linear(H, D_out)

    # lógica para calcular las salidas de la red
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


D_in, H, D_out = 784, 100, 10

model = ModeloPersonalizado(D_in, H, D_out)
print(model)

# verificar que el modelo recibe los datos en la forma correcta
x_prueba = torch.randn(500, D_in)
print(x_prueba)
outputs  = model(x_prueba)
outputs.shape

## 4. Función de Pérdida y Optimizador

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)  # Descenso de gradiente estocástico

## 5. Entrenamiento

In [ ]:
epochs   = 1500
log_each = 10
l = []
model.train()
for e in range(1, epochs + 1):

    # se calculan las predicciones con los parametros actuales
    y_pred = model(X_t)

    # se calcula el error entre la prediccion y el valor real
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # se limpian los gradientes del paso anterior para no acumularlos
    optimizer.zero_grad()

    # se calculan automaticamente todos los gradientes
    loss.backward()

    # se actualizan los parametros del modelo con los gradientes calculados
    optimizer.step()

    if not e % log_each:
        print(f'Epoch {e}/{epochs} Loss {np.mean(l):.5f}')

## 6. Gráfica de Convergencia del Costo

In [ ]:
pyplot.plot(np.arange(len(l)), l, lw=2)
pyplot.xlabel('Número de iteraciones')
pyplot.ylabel('Costo J')
pass

## 7. Evaluación del Modelo

In [ ]:
def evaluate(x):
    model.eval()
    y_pred = model(x)
    y_probas = torch.softmax(y_pred, dim=1)
    return torch.argmax(y_probas, axis=1)

pred_train = evaluate(X_t)
print('Precisión entrenamiento: {:.2f}%'.format(
    accuracy_score(y_train, pred_train.numpy()) * 100))

pred_test = evaluate(torch.from_numpy(X_test.astype(np.float32)).float())
print('Precisión validación   : {:.2f}%'.format(
    accuracy_score(y_test, pred_test.numpy()) * 100))

## 8. Predicciones finales

In [ ]:
import random

r, c = 3, 5
fig  = pyplot.figure(figsize=(2*c, 2*r))
for _r in range(r):
    for _c in range(c):
        pyplot.subplot(r, c, _r*c + _c + 1)
        ix     = random.randint(0, len(X_test)-1)
        img    = X_test[ix]
        y_pred = evaluate(torch.tensor([img]).float())[0]
        pyplot.imshow(img.reshape(28, 28), cmap='gray')
        pyplot.axis('off')
        pyplot.title(
            f'{y_test[ix]}/{y_pred}',
            color='green' if y_test[ix] == y_pred else 'red'
        )
pyplot.tight_layout()
pyplot.show()